In [1]:
# pip install agno langchain langchain-community langchain-text-splitters
# pip install langchain-qdrant qdrant-client fastembed
# ollama pull qwen2.5   # or qwen2.5:7b / 14b etc.

from agno.agent import Agent
from agno.knowledge.langchain import LangChainKnowledgeBase
from agno.models.ollama import Ollama
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from qdrant_client.http.exceptions import UnexpectedResponse


# ---------- 1) Load & chunk ----------
urls = ["https://blog.google/technology/developers/gemma-3/"]

docs = WebBaseLoader(urls).load()  # html → langchain Documents

splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=80)

chunks = splitter.split_documents(docs)


# ---------- 2) Embeddings ----------
# gte-large = 1024-dim; keep Qdrant vectors_config in sync
emb = FastEmbedEmbeddings(model_name="thenlper/gte-large")


# ---------- 3) Local Qdrant (on-disk) ----------
# Use path=":memory:" for RAM-only; use a folder path to persist on disk
client = QdrantClient(path="/tmp/qdrant_local")  # local mode

collection = "gemma3-rag"

try:
    client.get_collection(collection_name=collection)
except (UnexpectedResponse, ValueError):
    client.create_collection(
        collection_name=collection,
        vectors_config=VectorParams(size=1024, distance=Distance.COSINE)
    )


# ---------- 4) VectorStore & Retriever ----------
vstore = QdrantVectorStore(
    client=client,
    collection_name=collection,
    embedding=emb,
)

# Optional: add in batches to speed up first-time ingestion
vstore.add_documents(documents=chunks)

# Choose retriever mode; MMR gives quality-diversity tradeoff
retriever = vstore.as_retriever(
    search_type="mmr",      # "similarity" | "mmr"
    search_kwargs={"k": 6, "fetch_k": 20, "lambda_mult": 0.5}
)


# ---------- 5) Agno Knowledge + Agent ----------
kb = LangChainKnowledgeBase(retriever=retriever)

agent = Agent(
    model=Ollama(id="qwen2.5"),        # runs via local Ollama
    knowledge=kb,
    description="Answer using the knowledge base; cite sources when possible.",
    markdown=True,
    search_knowledge=True,  # force RAG before generation
)


q = "What new capabilities can developers use with Gemma 3?"

agent.print_response(q, stream=True)